# Weird States Analysis: socceraction Pipeline

This notebook demonstrates **semantically questionable but technically valid states** in the socceraction pipeline.

We construct minimal examples that expose brittleness in:
- Temporal discontinuity handling
- Credit assignment across possession chains
- Fixed probability injection for set pieces
- Grid discretization artifacts in xT

This is **not** a tutorial. It is a systems critique.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# socceraction imports
import socceraction.spadl as spadl
import socceraction.vaep as vaep
from socceraction.vaep import formula, labels, features as fs
from socceraction.xthreat import _get_cell_indexes

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Utility: Synthetic SPADL Constructor

We construct minimal SPADL action sequences to isolate specific failure modes.

This is not "fake" data—it is **schematic data** designed to test boundary conditions.

In [ ]:
def create_synthetic_actions(action_specs):
    """
    Create synthetic SPADL actions from specifications.
    
    Parameters
    ----------
    action_specs : list of dict
        Each dict specifies an action with keys: type_name, result_name, team_id, 
        time_seconds, start_x, start_y, end_x, end_y
    
    Returns
    -------
    pd.DataFrame
        SPADL-compliant actions dataframe
    """
    actions = []
    for i, spec in enumerate(action_specs):
        action = {
            'game_id': 1,
            'original_event_id': i,
            'action_id': i,
            'period_id': spec.get('period_id', 1),
            'time_seconds': spec['time_seconds'],
            'team_id': spec['team_id'],
            'player_id': spec.get('player_id', spec['team_id'] * 100 + i),
            'start_x': spec['start_x'],
            'start_y': spec['start_y'],
            'end_x': spec['end_x'],
            'end_y': spec['end_y'],
            'bodypart_id': spec.get('bodypart_id', 0),  # foot
            'bodypart_name': spec.get('bodypart_name', 'foot'),
            'type_id': spadl.config.actiontypes.index(spec['type_name']),
            'type_name': spec['type_name'],
            'result_id': spadl.config.results.index(spec['result_name']),
            'result_name': spec['result_name'],
        }
        actions.append(action)
    return pd.DataFrame(actions)

def compute_vaep_naive(actions):
    """
    Compute VAEP values using a naive model that assumes uniform scoring/conceding probabilities.
    
    This is not a trained model—it uses fixed probabilities to isolate formula behavior.
    """
    # Compute features (needed for game states)
    gamestates = fs.gamestates(actions, nb_prev_actions=3)
    
    # Assign naive probabilities based on action type and position
    # This is a simplified heuristic, not a trained model
    def naive_score_prob(action):
        if 'shot' in action['type_name']:
            return 0.10  # 10% chance from shots
        elif action['start_x'] > 80:  # attacking third
            return 0.05
        else:
            return 0.01
    
    def naive_concede_prob(action):
        if action['result_name'] == 'fail':  # turnover risk
            return 0.03
        else:
            return 0.01
    
    Pscores = pd.Series([naive_score_prob(row) for _, row in actions.iterrows()], index=actions.index)
    Pconcedes = pd.Series([naive_concede_prob(row) for _, row in actions.iterrows()], index=actions.index)
    
    # Compute VAEP values using the formula
    values = formula.value(actions, Pscores, Pconcedes)
    
    return values

---

## Weird State 1: Temporal Discontinuity Cliff

**Hypothesis**: The 10-second threshold for temporal continuity creates **cliff effects** where small time perturbations cause large valuation swings.

### Construction

We create two identical possession chains:
- **Chain A**: Actions separated by 9 seconds → continuity preserved
- **Chain B**: Actions separated by 11 seconds → prior state zeroed

The **only difference** is a 2-second delay, yet the valuation logic treats them as fundamentally different states.

In [ ]:
# Chain A: 9-second gap (below threshold)
chain_a = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 100.0,
     'start_x': 50, 'start_y': 34, 'end_x': 60, 'end_y': 34},
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 109.0,  # 9 sec gap
     'start_x': 60, 'start_y': 34, 'end_x': 70, 'end_y': 34},
    {'type_name': 'shot', 'result_name': 'success', 'team_id': 1, 'time_seconds': 112.0,
     'start_x': 70, 'start_y': 34, 'end_x': 105, 'end_y': 34},
])

# Chain B: 11-second gap (above threshold)
chain_b = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 100.0,
     'start_x': 50, 'start_y': 34, 'end_x': 60, 'end_y': 34},
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 111.0,  # 11 sec gap
     'start_x': 60, 'start_y': 34, 'end_x': 70, 'end_y': 34},
    {'type_name': 'shot', 'result_name': 'success', 'team_id': 1, 'time_seconds': 114.0,
     'start_x': 70, 'start_y': 34, 'end_x': 105, 'end_y': 34},
])

# Compute VAEP values
values_a = compute_vaep_naive(chain_a)
values_b = compute_vaep_naive(chain_b)

print("=" * 80)
print("WEIRD STATE 1: Temporal Discontinuity Cliff")
print("=" * 80)
print("\nChain A (9-second gap - continuous):")
print(pd.concat([chain_a[['time_seconds', 'type_name', 'result_name']], values_a], axis=1))

print("\nChain B (11-second gap - discontinuous):")
print(pd.concat([chain_b[['time_seconds', 'type_name', 'result_name']], values_b], axis=1))

print("\n" + "-" * 80)
print("Analysis:")
print("-" * 80)
print(f"Action 1 value difference: {values_a.iloc[1]['vaep_value'] - values_b.iloc[1]['vaep_value']:.6f}")
print("\nThe second pass in Chain B is valued against a ZEROED prior state,")
print("not the actual pre-stoppage state. This creates a discontinuity:")
print("  - If the 9-second chain reflects 'normal' play, the model captures state transitions.")
print("  - If the 11-second chain reflects a stoppage (e.g., injury), the model ERASES context.")
print("\nThe 10-second threshold is arbitrary. There is no semantic justification for this boundary.")
print("\nImplication: Small timing noise (e.g., logging delays) can cause large valuation swings.")

### Deeper Investigation: Time Delta Distribution

We examine what happens at the boundary by varying the gap from 8 to 12 seconds.

In [ ]:
# Construct chains with varying time gaps
time_gaps = np.arange(8, 13, 0.5)
action1_values = []

for gap in time_gaps:
    chain = create_synthetic_actions([
        {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 100.0,
         'start_x': 50, 'start_y': 34, 'end_x': 60, 'end_y': 34},
        {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 100.0 + gap,
         'start_x': 60, 'start_y': 34, 'end_x': 70, 'end_y': 34},
        {'type_name': 'shot', 'result_name': 'success', 'team_id': 1, 'time_seconds': 100.0 + gap + 3,
         'start_x': 70, 'start_y': 34, 'end_x': 105, 'end_y': 34},
    ])
    values = compute_vaep_naive(chain)
    action1_values.append(values.iloc[1]['vaep_value'])

# Plot the cliff effect
plt.figure(figsize=(10, 6))
plt.plot(time_gaps, action1_values, marker='o', linewidth=2, markersize=8)
plt.axvline(x=10, color='red', linestyle='--', linewidth=2, label='10-second threshold')
plt.xlabel('Time Gap (seconds)', fontsize=12)
plt.ylabel('VAEP Value of Second Pass', fontsize=12)
plt.title('Temporal Discontinuity Cliff Effect', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nObservation: The value function exhibits a DISCONTINUITY at exactly 10 seconds.")
print("This is a model artifact, not a reflection of football semantics.")

---

## Weird State 2: Credit Assignment Ambiguity

**Hypothesis**: The 10-action lookahead window assigns equal credit to all contributing actions, regardless of their causal contribution.

### Construction

We create a possession chain:
1. Safe back-pass in defensive third (low contribution)
2. Progressive pass to midfield (medium contribution)
3. Key assist pass to attacker (high contribution)
4. Shot → Goal

All three passes receive the **same binary label** (scores = True), despite vastly different causal roles.

In [ ]:
# Create possession chain leading to goal
possession_chain = create_synthetic_actions([
    # Action 0: Goalkeeper picks up ball
    {'type_name': 'keeper_pick_up', 'result_name': 'success', 'team_id': 1, 'time_seconds': 200.0,
     'start_x': 10, 'start_y': 34, 'end_x': 10, 'end_y': 34},
    # Action 1: Safe back-pass to defender
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 202.0,
     'start_x': 10, 'start_y': 34, 'end_x': 20, 'end_y': 40},
    # Action 2: Progressive pass to midfielder
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 204.0,
     'start_x': 20, 'start_y': 40, 'end_x': 50, 'end_y': 34},
    # Action 3: Key assist pass
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 206.0,
     'start_x': 50, 'start_y': 34, 'end_x': 90, 'end_y': 34},
    # Action 4: Shot → Goal
    {'type_name': 'shot', 'result_name': 'success', 'team_id': 1, 'time_seconds': 208.0,
     'start_x': 90, 'start_y': 34, 'end_x': 105, 'end_y': 34},
])

# Compute labels using the standard VAEP label function
goal_labels = labels.scores(possession_chain, nr_actions=10)

print("=" * 80)
print("WEIRD STATE 2: Credit Assignment Ambiguity")
print("=" * 80)
print("\nPossession chain with labels:")
print(pd.concat([
    possession_chain[['action_id', 'time_seconds', 'type_name', 'start_x', 'result_name']], 
    goal_labels
], axis=1))

print("\n" + "-" * 80)
print("Analysis:")
print("-" * 80)
print("All actions receive label = True (goal scored within 10 actions).")
print("\nBut their causal contributions differ:")
print("  - Action 0 (keeper pick-up): Neutral—just restarting play.")
print("  - Action 1 (back-pass):      Low causal contribution.")
print("  - Action 2 (progressive):     Medium contribution—advances the ball.")
print("  - Action 3 (assist):          High contribution—creates the chance.")
print("  - Action 4 (shot):            Direct contribution—scores the goal.")
print("\nThe labeling function treats them EQUALLY. This is correct for classification,")
print("but semantically problematic for valuation:")
print("  - A classifier learns P(goal | state), averaging over all paths.")
print("  - A valuation system needs P(goal | action), which requires credit assignment.")
print("\nThe model conflates 'preceded a goal' with 'contributed to a goal'.")

### Stress Test: Long Possession Chains

What happens when the possession chain is exactly 10 actions long vs. 11 actions?

In [ ]:
def create_chain_with_n_actions(n_actions):
    """Create a possession chain of n actions ending in a goal."""
    specs = []
    for i in range(n_actions - 1):
        specs.append({
            'type_name': 'pass',
            'result_name': 'success',
            'team_id': 1,
            'time_seconds': 300.0 + i * 2,
            'start_x': min(10 + i * 8, 95),
            'start_y': 34,
            'end_x': min(18 + i * 8, 100),
            'end_y': 34,
        })
    # Final action: shot → goal
    specs.append({
        'type_name': 'shot',
        'result_name': 'success',
        'team_id': 1,
        'time_seconds': 300.0 + (n_actions - 1) * 2,
        'start_x': 95,
        'start_y': 34,
        'end_x': 105,
        'end_y': 34,
    })
    return create_synthetic_actions(specs)

# Chain with 10 actions
chain_10 = create_chain_with_n_actions(10)
labels_10 = labels.scores(chain_10, nr_actions=10)

# Chain with 11 actions
chain_11 = create_chain_with_n_actions(11)
labels_11 = labels.scores(chain_11, nr_actions=10)

print("\nChain with 10 actions (all labeled):")
print(f"  First action label: {labels_10.iloc[0]['scores']}")
print(f"  Last action label:  {labels_10.iloc[-1]['scores']}")

print("\nChain with 11 actions (first action outside window):")
print(f"  First action label: {labels_11.iloc[0]['scores']}  <-- EXCLUDED")
print(f"  Second action label: {labels_11.iloc[1]['scores']}")
print(f"  Last action label:   {labels_11.iloc[-1]['scores']}")

print("\n" + "-" * 80)
print("The first action in the 11-action chain is EXCLUDED from the goal label.")
print("This creates a cliff effect: a single additional action in the chain")
print("causes the earliest action to be treated as NOT contributing to the goal.")
print("\nThis is a consequence of the fixed 10-action window. There is no theoretical")
print("justification for this cutoff—it is an arbitrary hyperparameter.")

---

## Weird State 3: Fixed Probability Injection

**Hypothesis**: Hardcoded probabilities for penalties (79.2%) and corners (4.65%) create **discontinuities** between learned and injected values.

### Construction

We create two scenarios:
1. A free kick just outside the penalty box (learned probability)
2. A penalty kick (hardcoded probability = 0.792453)

The **spatial difference** is minimal (<1 meter), but the valuation logic is fundamentally different.

In [ ]:
# Scenario 1: Free kick just outside the box
freekick_scenario = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 400.0,
     'start_x': 80, 'start_y': 34, 'end_x': 87, 'end_y': 34},
    {'type_name': 'shot_freekick', 'result_name': 'success', 'team_id': 1, 'time_seconds': 402.0,
     'start_x': 87, 'start_y': 34, 'end_x': 105, 'end_y': 34},  # Just outside box
])

# Scenario 2: Penalty kick
penalty_scenario = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 400.0,
     'start_x': 80, 'start_y': 34, 'end_x': 90, 'end_y': 34},
    {'type_name': 'shot_penalty', 'result_name': 'success', 'team_id': 1, 'time_seconds': 402.0,
     'start_x': 94, 'start_y': 34, 'end_x': 105, 'end_y': 34},  # Penalty spot
])

# Compute values
values_freekick = compute_vaep_naive(freekick_scenario)
values_penalty = compute_vaep_naive(penalty_scenario)

print("=" * 80)
print("WEIRD STATE 3: Fixed Probability Injection")
print("=" * 80)
print("\nScenario 1: Free kick just outside box")
print(pd.concat([freekick_scenario[['type_name', 'start_x']], values_freekick], axis=1))

print("\nScenario 2: Penalty kick")
print(pd.concat([penalty_scenario[['type_name', 'start_x']], values_penalty], axis=1))

print("\n" + "-" * 80)
print("Analysis:")
print("-" * 80)
print("The penalty's prior probability is HARDCODED to 0.792453 in formula.py:63.")
print("This value is INJECTED, not learned from data.")
print("\nConsequences:")
print("  1. The penalty's value is DECOUPLED from the model's learned probabilities.")
print("  2. If the training data has a different penalty conversion rate, the model")
print("     will produce INCONSISTENT valuations.")
print("  3. The hardcoded constant ignores context:")
print("     - Penalty taker skill")
print("     - Goalkeeper quality")
print("     - Match pressure (e.g., 90+5' to equalize)")
print("\nThis is a MODEL DECISION, not a reflection of data. It creates a discontinuity:")
print("  - Free kicks are valued by the classifier.")
print("  - Penalties are valued by a constant.")
print("\nThe boundary is ARBITRARY: why are penalties special, but not free kicks?")

### Corner Discontinuity

Similarly, corners have a hardcoded probability of 0.0465 (4.65%).

In [ ]:
# Scenario: Cross from wing vs. corner
cross_scenario = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 500.0,
     'start_x': 80, 'start_y': 34, 'end_x': 95, 'end_y': 5},
    {'type_name': 'cross', 'result_name': 'success', 'team_id': 1, 'time_seconds': 502.0,
     'start_x': 95, 'start_y': 5, 'end_x': 100, 'end_y': 34},  # Cross from wing
])

corner_scenario = create_synthetic_actions([
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 500.0,
     'start_x': 80, 'start_y': 34, 'end_x': 95, 'end_y': 2},
    {'type_name': 'corner_crossed', 'result_name': 'success', 'team_id': 1, 'time_seconds': 502.0,
     'start_x': 105, 'start_y': 0, 'end_x': 100, 'end_y': 34},  # Corner
])

values_cross = compute_vaep_naive(cross_scenario)
values_corner = compute_vaep_naive(corner_scenario)

print("\nCross from wing:")
print(pd.concat([cross_scenario[['type_name', 'start_x', 'start_y']], values_cross], axis=1))

print("\nCorner:")
print(pd.concat([corner_scenario[['type_name', 'start_x', 'start_y']], values_corner], axis=1))

print("\nThe corner's prior is hardcoded to 0.046500 (formula.py:67).")
print("This creates the same discontinuity as penalties: learned vs. injected values.")

---

## Weird State 4: Grid Discretization Artifacts (xT)

**Hypothesis**: The xT framework's 12×16 grid creates **boundary artifacts** where infinitesimal spatial displacements cause discrete value jumps.

### Construction

We create two actions with nearly identical coordinates, but on opposite sides of a grid boundary.

In [ ]:
from socceraction.xthreat import _get_cell_indexes
import socceraction.spadl.config as spadlcfg

# Define two actions with minimal spatial separation
x1, y1 = 52.4, 34.0  # Just below grid boundary
x2, y2 = 52.6, 34.0  # Just above grid boundary

# Convert to grid cells
xi1, yj1 = _get_cell_indexes(pd.Series([x1]), pd.Series([y1]))
xi2, yj2 = _get_cell_indexes(pd.Series([x2]), pd.Series([y2]))

print("=" * 80)
print("WEIRD STATE 4: Grid Discretization Artifacts (xT)")
print("=" * 80)
print(f"\nAction A: (x={x1:.1f}, y={y1:.1f}) → Grid cell ({xi1.iloc[0]}, {yj1.iloc[0]})")
print(f"Action B: (x={x2:.1f}, y={y2:.1f}) → Grid cell ({xi2.iloc[0]}, {yj2.iloc[0]})")

if xi1.iloc[0] != xi2.iloc[0] or yj1.iloc[0] != yj2.iloc[0]:
    print("\n⚠️  DIFFERENT CELLS despite 20cm separation!")
else:
    print("\n✓ Same cell")

print("\n" + "-" * 80)
print("Analysis:")
print("-" * 80)
print(f"Grid dimensions: {16} (length) × {12} (width)")
print(f"Cell size: {spadlcfg.field_length / 16:.2f}m × {spadlcfg.field_width / 12:.2f}m")
print("\nEach cell is ~6.6m × 5.7m. Within-cell variance is COLLAPSED:")
print("  - A pass from (52.0, 34.0) to (58.0, 34.0) may be in the SAME cell.")
print("  - A pass from (52.4, 34.0) to (52.6, 34.0) may be in DIFFERENT cells.")
print("\nThis discretization is a LOSSY COMPRESSION. Semantically distinct actions")
print("are treated identically, while near-identical actions are treated differently.")
print("\nThe grid resolution (12×16) is a hyperparameter, not a principled choice.")

### Boundary Sensitivity Test

We vary the x-coordinate from 50 to 55 meters and observe cell transitions.

In [ ]:
# Sweep x-coordinate across midfield
x_coords = np.linspace(50, 55, 100)
y_coord = 34.0

cells_x = []
for x in x_coords:
    xi, _ = _get_cell_indexes(pd.Series([x]), pd.Series([y_coord]))
    cells_x.append(xi.iloc[0])

plt.figure(figsize=(10, 6))
plt.plot(x_coords, cells_x, linewidth=2)
plt.xlabel('X Coordinate (meters)', fontsize=12)
plt.ylabel('Grid Cell (x-dimension)', fontsize=12)
plt.title('xT Grid Cell Assignment: Boundary Discontinuities', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe step function reveals discrete jumps at cell boundaries.")
print("These jumps are MODEL ARTIFACTS, not semantic boundaries in football space.")
print("\nConsequence: Small positional noise (e.g., GPS uncertainty) can cause")
print("different threat assessments for the same action.")

---

## Weird State 5: Possession Flip Valuation

**Hypothesis**: When possession changes hands, the value formula **inverts** the scores/concedes probabilities. This creates counterintuitive valuations for defensive actions.

### Construction

We create a sequence where Team A loses possession to Team B via a tackle.

In [ ]:
# Possession flip sequence
possession_flip = create_synthetic_actions([
    # Team A attacks
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 1, 'time_seconds': 600.0,
     'start_x': 70, 'start_y': 34, 'end_x': 80, 'end_y': 34},
    # Team B tackles
    {'type_name': 'tackle', 'result_name': 'success', 'team_id': 2, 'time_seconds': 601.0,
     'start_x': 80, 'start_y': 34, 'end_x': 75, 'end_y': 34},
    # Team B counter-attacks
    {'type_name': 'pass', 'result_name': 'success', 'team_id': 2, 'time_seconds': 603.0,
     'start_x': 75, 'start_y': 34, 'end_x': 60, 'end_y': 34},
    {'type_name': 'shot', 'result_name': 'success', 'team_id': 2, 'time_seconds': 605.0,
     'start_x': 60, 'start_y': 34, 'end_x': 0, 'end_y': 34},
])

values_flip = compute_vaep_naive(possession_flip)

print("=" * 80)
print("WEIRD STATE 5: Possession Flip Valuation")
print("=" * 80)
print("\nPossession sequence:")
print(pd.concat([
    possession_flip[['action_id', 'team_id', 'type_name', 'result_name']], 
    values_flip
], axis=1))

print("\n" + "-" * 80)
print("Analysis:")
print("-" * 80)
print("The tackle (action 1) is performed by Team B, which FLIPS the perspective.")
print("\nIn formula.py:48-49, the value is computed as:")
print("  sameteam = (_prev(actions.team_id) == actions.team_id)")
print("  prev_scores = _prev(scores) * sameteam + _prev(concedes) * (~sameteam)")
print("\nWhen sameteam=False (possession flip):")
print("  - The PREVIOUS team's concede probability becomes the CURRENT team's score probability.")
print("  - This is CORRECT for the new possessing team...")
print("  - ...but creates AMBIGUITY in attributing value to the transition action.")
print("\nQuestion: Who gets credit for a successful tackle?")
print("  - The defender (Team B) who executed the tackle?")
print("  - Or should we penalize the attacker (Team A) who lost the ball?")
print("\nThe model attributes value to the tackle itself, but does NOT penalize")
print("the preceding action (Team A's pass) for being susceptible to a tackle.")
print("\nThis asymmetry is a MODELING CHOICE, not a reflection of causality.")

---

## Synthesis: Where Semantics End

### Summary of Findings

| Weird State | Root Cause | Semantic Impact |
|-------------|------------|----------------|
| **WS1**: Temporal cliff | Hardcoded 10-second threshold | Arbitrary discontinuity; context erasure |
| **WS2**: Credit ambiguity | Binary labels over 10-action window | Equal credit to unequal contributions |
| **WS3**: Fixed probabilities | Injected constants (penalties, corners) | Learned vs. hardcoded value incoherence |
| **WS4**: Grid discretization | 12×16 spatial quantization | Boundary artifacts; positional noise sensitivity |
| **WS5**: Possession flip | Score/concede probability inversion | Asymmetric credit for turnovers |

### Implications

These weird states are **not bugs**—they are **consequences of design decisions**:

1. **Temporal Discretization**: The 10-second threshold is a pragmatic choice to handle stoppages, but it introduces artifacts.
   
2. **Markovian Truncation**: The 3-action + 10-action lookahead windows are hyperparameters, not principled choices grounded in football semantics.

3. **Fixed Priors**: Injecting hardcoded probabilities for set pieces creates a **two-tier valuation system** (learned vs. constant).

4. **Spatial Quantization**: xT's grid is a lossy compression that sacrifices precision for computational efficiency.

5. **Attribution Ambiguity**: The model values **state transitions**, not **actions**. Causality is implicit, not explicit.

### Potential Invariants

To detect these weird states in production:

1. **WS1**: Assert `|time_delta| < threshold` or flag stoppage contexts explicitly.
2. **WS2**: Weight labels by inverse distance to the goal (e.g., exponential decay).
3. **WS3**: Compare hardcoded constants to dataset statistics; log when they diverge.
4. **WS4**: Perturb coordinates by ±ε and check for cell transitions (sensitivity test).
5. **WS5**: Explicitly model credit for both the gaining and losing team in turnovers.

### Conclusion

The socceraction pipeline is **technically sound** but **semantically underconstrained**. It produces valid outputs for "normal" football, but its assumptions break down at the edges:

- Extreme temporal gaps
- Long possession chains
- High-variance contexts (scoreline, red cards)
- Boundary conditions (set pieces, grid edges)

This audit provides a foundation for:
- **Robustness testing**: Probing these failure modes systematically
- **Model extensions**: Adding contextual features to relax stationarity
- **Calibration**: Detecting when assumptions are violated

The model's semantics **end** where its assumptions **begin**.